# 01 — Preparação e Qualidade dos Dados

## Objetivo

Este notebook realiza:

1. Carregamento dos dados PEDE 2022, 2023 e 2024;
2. Auditoria inicial da qualidade dos dados;
3. Padronização dos nomes e tipos das variáveis;
4. Padronização das fases e classificações;
5. Validação do IAN e da defasagem;
6. Construção da base longitudinal;
7. Criação das variáveis necessárias para análises futuras;
8. Exportação da base processada.

In [7]:
from pathlib import Path

import pandas as pd
import numpy as np
import re
import unicodedata

# CAMINHOS DO PROJETO

ROOT = Path.cwd()

# Caso o notebook esteja sendo executado dentro de /notebooks

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

ARQUIVO = ROOT / "data" / "raw" / "BASE DE DADOS PEDE 2024 - DATATHON.xlsx"

PASTA_PROCESSADOS = ROOT / "data" / "processed"
PASTA_TABELAS = ROOT / "outputs" / "tables"
PASTA_FIGURAS = ROOT / "outputs" / "figures"

PASTA_PROCESSADOS.mkdir(parents=True, exist_ok=True)
PASTA_TABELAS.mkdir(parents=True, exist_ok=True)
PASTA_FIGURAS.mkdir(parents=True, exist_ok=True)


# CARREGAMENTO

bases = pd.read_excel(
    ARQUIVO,
    sheet_name=["PEDE2022", "PEDE2023", "PEDE2024"]
)

df22 = bases["PEDE2022"]
df23 = bases["PEDE2023"]
df24 = bases["PEDE2024"]


# AUDITORIA INICIAL

for ano, df in {
    2022: df22,
    2023: df23,
    2024: df24
}.items():

    print("=" * 70)
    print(f"PEDE {ano}")
    print("=" * 70)

    print(f"Linhas: {len(df):,}")
    print(f"Colunas: {df.shape[1]}")
    print(f"RAs únicos: {df['RA'].nunique()}")
    print(f"RAs duplicados: {df['RA'].duplicated().sum()}")

    print("\n10 colunas com maior percentual de valores ausentes:")

    missing = (
        df.isna()
        .mean()
        .mul(100)
        .sort_values(ascending=False)
        .head(10)
        .round(2)
    )

    print(missing)
    print()

PEDE 2022
Linhas: 860
Colunas: 42
RAs únicos: 860
RAs duplicados: 0

10 colunas com maior percentual de valores ausentes:
Inglês        67.09
Rec Av4       65.58
Avaliador4    63.95
Pedra 20      62.44
Pedra 21      46.28
Avaliador3    37.91
Matem          0.23
Portug         0.23
Fase           0.00
RA             0.00
dtype: float64

PEDE 2023
Linhas: 1,014
Colunas: 48
RAs únicos: 1014
RAs duplicados: 0

10 colunas com maior percentual de valores ausentes:
Pedra 23          100.0
Rec Psicologia    100.0
Indicado          100.0
Atingiu PV        100.0
Destaque IEG      100.0
Destaque IDA      100.0
Destaque IPV      100.0
Destaque IPV.1    100.0
Rec Av2           100.0
Rec Av3           100.0
dtype: float64

PEDE 2024
Linhas: 1,156
Colunas: 50
RAs únicos: 1156
RAs duplicados: 0

10 colunas com maior percentual de valores ausentes:
Cg                100.0
Cf                100.0
Atingiu PV        100.0
Indicado          100.0
Destaque IEG      100.0
Rec Psicologia    100.0
Rec Av1     

In [8]:
# FUNÇÕES DE PADRONIZAÇÃO

def extrair_fase(valor):
    """
    Converte diferentes formatos de fase para um número comparável.

    Exemplos:
    ALFA   -> 0
    FASE 1 -> 1
    2K     -> 2
    7E     -> 7
    """

    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip().upper()

    if texto == "ALFA":
        return 0

    numero = re.search(r"\d+", texto)

    if numero:
        return int(numero.group())

    return np.nan


def normalizar_pedra(valor):
    """
    Padroniza a classificação Pedra.
    """

    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip()

    texto_sem_acento = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    ).upper()

    mapa = {
        "QUARTZO": "Quartzo",
        "AGATA": "Ágata",
        "AMETISTA": "Ametista",
        "TOPAZIO": "Topázio"
    }

    return mapa.get(texto_sem_acento, np.nan)


def classificar_defasagem(d):
    """
    Classificação baseada na defasagem entre fase efetiva e fase ideal.
    """

    if pd.isna(d):
        return np.nan

    if d >= 0:
        return "Em fase / adiantado"

    if d >= -2:
        return "Defasagem moderada"

    return "Defasagem severa"


def calcular_ian(d):
    """
    Reconstrói o IAN apenas para validar o dado existente.
    """

    if pd.isna(d):
        return np.nan

    if d >= 0:
        return 10

    if d >= -2:
        return 5

    return 2.5


# PADRONIZAÇÃO POR ANO

def preparar_ano(df, ano):

    df = df.copy()

    if ano == 2022:

        mapa = {
            "RA": "RA",
            "Fase": "fase_raw",
            "Idade 22": "idade",
            "Gênero": "genero",
            "Ano ingresso": "ano_ingresso",
            "Instituição de ensino": "instituicao",
            "Pedra 22": "pedra",
            "INDE 22": "inde",
            "IAA": "iaa",
            "IEG": "ieg",
            "IPS": "ips",
            "IDA": "ida",
            "Matem": "mat",
            "Portug": "por",
            "Inglês": "ing",
            "IPV": "ipv",
            "IAN": "ian",
            "Fase ideal": "fase_ideal",
            "Defas": "defasagem"
        }

    else:

        mapa = {
            "RA": "RA",
            "Fase": "fase_raw",
            "Idade": "idade",
            "Gênero": "genero",
            "Ano ingresso": "ano_ingresso",
            "Instituição de ensino": "instituicao",
            f"Pedra {ano}": "pedra",
            f"INDE {ano}": "inde",
            "IAA": "iaa",
            "IEG": "ieg",
            "IPS": "ips",
            "IPP": "ipp",
            "IDA": "ida",
            "Mat": "mat",
            "Por": "por",
            "Ing": "ing",
            "IPV": "ipv",
            "IAN": "ian",
            "Fase Ideal": "fase_ideal",
            "Defasagem": "defasagem"
        }

    df = df.rename(columns=mapa)

    colunas_finais = [
        "RA",
        "fase_raw",
        "idade",
        "genero",
        "ano_ingresso",
        "instituicao",
        "pedra",
        "inde",
        "iaa",
        "ieg",
        "ips",
        "ipp",
        "ida",
        "mat",
        "por",
        "ing",
        "ipv",
        "ian",
        "fase_ideal",
        "defasagem"
    ]

    # Cria colunas inexistentes, como IPP em 2022
    for coluna in colunas_finais:
        if coluna not in df.columns:
            df[coluna] = np.nan

    df = df[colunas_finais]

    df["ano"] = ano

    # Padronização da fase
    df["fase_num"] = df["fase_raw"].apply(extrair_fase)

    # Padronização da Pedra
    df["pedra"] = df["pedra"].apply(normalizar_pedra)

    # Colunas que devem ser numéricas
    colunas_numericas = [
        "idade",
        "ano_ingresso",
        "inde",
        "iaa",
        "ieg",
        "ips",
        "ipp",
        "ida",
        "mat",
        "por",
        "ing",
        "ipv",
        "ian",
        "defasagem",
        "fase_num"
    ]

    for coluna in colunas_numericas:
        df[coluna] = pd.to_numeric(
            df[coluna],
            errors="coerce"
        )

    # Quantidade de anos do aluno na Passos Mágicos
    df["anos_na_pm"] = df["ano"] - df["ano_ingresso"]

    # Classificação da defasagem
    df["faixa_defasagem"] = df["defasagem"].apply(
        classificar_defasagem
    )

    # Aluno atualmente defasado
    df["risco_atual"] = (
        df["defasagem"] < 0
    ).astype("Int64")

    # Recalcula o IAN para checagem de consistência
    df["ian_calculado"] = df["defasagem"].apply(calcular_ian)

    df["ian_confere"] = np.isclose(
        df["ian"],
        df["ian_calculado"],
        equal_nan=True
    )

    return df


base22 = preparar_ano(df22, 2022)
base23 = preparar_ano(df23, 2023)
base24 = preparar_ano(df24, 2024)


# JUNÇÃO DOS ANOS

base = pd.concat(
    [base22, base23, base24],
    ignore_index=True
)

base = base.sort_values(
    ["RA", "ano"]
).reset_index(drop=True)


# VARIÁVEIS DO ANO SEGUINTE

base["ano_prox"] = (
    base.groupby("RA")["ano"]
    .shift(-1)
)

variaveis_futuras = [
    "defasagem",
    "ida",
    "ieg",
    "ips",
    "ipp",
    "ipv",
    "ian",
    "inde",
    "pedra"
]

for coluna in variaveis_futuras:

    base[f"{coluna}_prox"] = (
        base.groupby("RA")[coluna]
        .shift(-1)
    )

    # Não permite que 2022 seja ligado diretamente a 2024
    mascara_nao_consecutiva = (
        base["ano_prox"] != base["ano"] + 1
    )

    base.loc[
        mascara_nao_consecutiva,
        f"{coluna}_prox"
    ] = np.nan


# TARGET PARA MACHINE LEARNING

base["risco_prox_ano"] = pd.NA

mascara_target = base["defasagem_prox"].notna()

base.loc[
    mascara_target,
    "risco_prox_ano"
] = (
    base.loc[
        mascara_target,
        "defasagem_prox"
    ] < 0
).astype(int)

base["risco_prox_ano"] = (
    base["risco_prox_ano"]
    .astype("Int64")
)


# VARIAÇÕES ENTRE ANOS

base["delta_ida"] = (
    base["ida_prox"] - base["ida"]
)

base["delta_ieg"] = (
    base["ieg_prox"] - base["ieg"]
)

base["delta_inde"] = (
    base["inde_prox"] - base["inde"]
)


# VALIDAÇÕES

print("Registros por ano:")
print(base.groupby("ano").size())

print("\nIAN validado:")
print(
    base.groupby("ano")["ian_confere"]
    .mean()
    .mul(100)
    .round(2)
)

print("\nObservações disponíveis para previsão:")
print(
    base[
        base["risco_prox_ano"].notna()
    ]
    .groupby("ano")
    .size()
)

print("\nAlunos presentes nos três anos:")

ras22 = set(base22["RA"])
ras23 = set(base23["RA"])
ras24 = set(base24["RA"])

print(
    len(ras22 & ras23 & ras24)
)

Registros por ano:
ano
2022     860
2023    1014
2024    1156
dtype: int64

IAN validado:
ano
2022    100.0
2023    100.0
2024    100.0
Name: ian_confere, dtype: float64

Observações disponíveis para previsão:
ano
2022    600
2023    765
dtype: int64

Alunos presentes nos três anos:
468


In [9]:
# EXPORTAÇÃO DA BASE PROCESSADA

arquivo_saida = (
    PASTA_PROCESSADOS /
    "base_longitudinal.csv"
)

base.to_csv(
    arquivo_saida,
    index=False,
    encoding="utf-8-sig"
)

print(f"Base salva em: {arquivo_saida}")
print(f"Total de registros: {len(base):,}")
print(f"Total de colunas: {base.shape[1]}")

Base salva em: d:\aaDev\DATATHON-PASSOS-MAGICOS\data\processed\base_longitudinal.csv
Total de registros: 3,030
Total de colunas: 41


In [10]:
# CHECKLIST FINAL DE QUALIDADE

print("=" * 70)
print("CHECKLIST FINAL DA BASE PROCESSADA")
print("=" * 70)

# 1. RA + ano devem ser únicos

duplicados = base.duplicated(
    subset=["RA", "ano"]
).sum()

print(f"\nDuplicidades RA + ano: {duplicados}")


# 2. Quantidade de registros por ano

print("\nRegistros por ano:")

print(
    base["ano"]
    .value_counts()
    .sort_index()
)


# 3. Intervalos dos principais indicadores

indicadores = [
    "ian",
    "ida",
    "ieg",
    "iaa",
    "ips",
    "ipp",
    "ipv",
    "inde"
]

print("\nIntervalos dos indicadores:")

display(
    base[indicadores]
    .agg(["min", "max", "mean", "median"])
    .T
    .round(2)
)


# 4. Dados ausentes nos indicadores principais

print("\nPercentual de dados ausentes:")

ausentes = (
    base[indicadores]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

display(ausentes.to_frame("ausentes_%"))


# 5. Distribuição do target do modelo

print("\nDistribuição do risco do próximo ano:")

target = (
    base[
        base["risco_prox_ano"].notna()
    ]
    .groupby("ano")["risco_prox_ano"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

display(target)


# 6. Número de observações para Machine Learning

print("\nObservações disponíveis para Machine Learning:")

display(
    base[
        base["risco_prox_ano"].notna()
    ]
    .groupby("ano")
    .size()
    .rename("observacoes")
)


# RESULTADO FINAL

if duplicados == 0:
    print("\n✅ Base pronta para análise e Machine Learning.")
else:
    print("\n⚠️ Existem duplicidades que precisam ser verificadas.")

CHECKLIST FINAL DA BASE PROCESSADA

Duplicidades RA + ano: 0

Registros por ano:
ano
2022     860
2023    1014
2024    1156
Name: count, dtype: int64

Intervalos dos indicadores:


,min,max,mean,median
ian,2.50,10.00,7.18,5.00
ida,0.00,10.00,6.38,6.67
ieg,0.00,10.00,7.95,8.60
iaa,0.00,10.00,7.92,8.75
ips,2.50,10.00,6.29,7.50
ipp,2.50,10.00,7.56,7.50
ipv,2.50,10.01,7.55,7.58
inde,3.03,9.53,7.27,7.39



Percentual de dados ausentes:


,ausentes_%
ipp,34.26
inde,6.11
ida,5.87
ipv,5.87
ips,5.64
iaa,5.45
ieg,2.51
ian,0.00



Distribuição do risco do próximo ano:


ano   risco_prox_ano
2022  1                  61.0
      0                  39.0
2023  0                 59.74
      1                 40.26
Name: proportion, dtype: Float64


Observações disponíveis para Machine Learning:


ano
2022    600
2023    765
Name: observacoes, dtype: int64


✅ Base pronta para análise e Machine Learning.
